In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

KeyboardInterrupt: 

In [ ]:
print('Loading data...')
# Load smaller datasets into memory
books = pd.read_csv('data/books.csv')
games = pd.read_csv('data/games.csv')
movies = pd.read_csv('data/movies.csv')

books['media_type'] = 'book'
games['media_type'] = 'game'
movies['media_type'] = 'movie'

# Process shows.csv in chunks
chunk_list = []
for chunk in pd.read_csv('data/shows.csv', chunksize=10000):
    chunk['media_type'] = 'show'
    chunk_list.append(chunk)

shows = pd.concat(chunk_list, ignore_index=True)

# Concatenate all dataframes
df = pd.concat([books, games, movies, shows], ignore_index=True)
print('Data loaded and concatenated.')

# Feature engineering
print('Performing feature engineering...')
df['tags'] = df['overview'].fillna('') + ' ' + df['genres'].fillna('')
print('Feature engineering complete.')

# Save the master dataframe
print('Saving master dataframe...')
pickle.dump(df, open('master_df.pkl', 'wb'))
print('Master dataframe saved as master_df.pkl')

In [ ]:
# TF-IDF Vectorization
print('Performing TF-IDF vectorization...')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['tags'])
print('TF-IDF vectorization complete.')

In [ ]:
# Compute and save cosine similarity matrix in chunks
print('Computing and saving similarity matrix in chunks...')
if not os.path.exists('similarity_chunks'):
    os.makedirs('similarity_chunks')

chunk_size = 5000
for i in range(0, tfidf_matrix.shape[0], chunk_size):
    chunk = tfidf_matrix[i:i+chunk_size]
    sim_chunk = cosine_similarity(chunk, tfidf_matrix)
    with open(f'similarity_chunks/sim_chunk_{i}.pkl', 'wb') as f:
        pickle.dump(sim_chunk, f)
    print(f'Saved chunk {i} to {i+chunk_size}')
print('Similarity matrix saved in chunks.')